# Faster R-CNN on UA-DETRAC

Fine tunes Faster R-CNN and exports detections using the same scoring setup as the YOLO baseline. Set SMOKE_TEST to True for a quick test.

## 1. CONFIG

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/DATA255
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1bo0ITDfSqcaoFTjJXtJVrUDLpPS9E0SS/DATA255
annotations.csv  outputs  processed.zip  yolov10s-p2.yaml  yolov10s.pt


In [ ]:
import sys, io, json, time, random, shutil, contextlib, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from PIL import Image
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

# set root directory to the parent of notebooks/ if we are in notebooks/, otherwise cwd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

SMOKE_TEST = False   # True = quick test, False = real run

if IN_COLAB:
    COLAB_ZIP = Path('/content/drive/MyDrive/DATA255/processed.zip')     # zip of traffic-monitoring/data/processed
    DATA_DIR = Path('/content/processed')                               # where the zip is extracted
    OUTPUT_DIR = Path('/content/drive/MyDrive/DATA255/outputs/faster_rcnn')
else:
    DATA_DIR = ROOT / 'traffic-monitoring' / 'data' / 'processed'
    OUTPUT_DIR = ROOT / 'outputs' / 'faster_rcnn'

MODEL = 'fasterrcnn_resnet50_fpn_v2'   # torchvision, COCO weights
EPOCHS = 12            # torchvision's '1x' schedule length; checkpoint chosen on val
PATIENCE = None        # not used: every epoch is scored on val and the best one is kept
IMGSZ = 640            # only used by the shared (YOLO) helper; Faster R-CNN runs at native 960x540
BATCH = 4              # images per step (T4, 16 GB); lower to 2 if out of memory
LR = 0.01              # SGD, momentum 0.9, weight decay 1e-4, 500-step warm-up, x0.1 at epochs 8 and 11
MIN_SIZE, MAX_SIZE = 540, 960   # = UA-DETRAC frame size, so images are not resized
MAX_TRAIN_IMAGES = None   # i.e. 6000 if one epoch is too slow (random, fixed by SEED)
VAL_IMAGES = 600       # fixed random val subset scored after every epoch to pick the checkpoint
WORKERS = 2 if torch.cuda.is_available() else 0   # 0 on Mac (notebook defined datasets can't be sent to worker processes there)
SEED = 0
FRACTION = 1.0

EVAL_CONF = 0.001
COUNT_CONF = 0.25
IOU_MATCH = 0.5
SMOKE_TEST_IMAGES_PER_WEATHER = 10

if SMOKE_TEST:
    EPOCHS = 1
    FRACTION = 0.1
    MAX_TRAIN_IMAGES = 200
    VAL_IMAGES = 50
    OUTPUT_DIR = OUTPUT_DIR / 'smoke_test'

if torch.cuda.is_available():
    DEVICE = 0
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

# mapped COCO class IDs to our class IDs (0=car, 1=bus, 2=others)
COCO_TO_OURS = {2: 0,   # car   -> car
                5: 1,   # bus   -> bus
                7: 3}   # truck -> others

random.seed(SEED)
np.random.seed(SEED)
if not IN_COLAB:   
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'SMOKE_TEST={SMOKE_TEST}  DEVICE={DEVICE}  EPOCHS={EPOCHS}  FRACTION={FRACTION}')
print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

SMOKE_TEST=False  DEVICE=0  EPOCHS=12  FRACTION=1.0
DATA_DIR  : /content/processed
OUTPUT_DIR: /content/drive/MyDrive/DATA255/outputs/faster_rcnn


## 2. SETUP

For Colab: mount Google Drive and copy the P2 model and processed dataset files, skipped if local

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pycocotools'], check=True)
    if not DATA_DIR.exists():
        shutil.unpack_archive(COLAB_ZIP, DATA_DIR.parent)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    print('Not on Colab: skipping')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Load the dataset information and image lists.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Read the original data.yaml (class names + split folders)
with open(DATA_DIR / 'data.yaml') as f:
    original_yaml = yaml.safe_load(f)
names = original_yaml['names']
CLASS_NAMES = list(names) if isinstance(names, list) else [names[i] for i in sorted(names)]
NUM_CLASSES = len(CLASS_NAMES)
SPLITS = ['train', 'val', 'test']

# Dataset view: OUTPUT_DIR/dataset_view/{images,labels}/{split} are symlinks to the real folders
# On Colab keep the symlink view on the local disk: links stored on Drive make every image read go through Drive
VIEW_DIR = Path('/content/dataset_view') if IN_COLAB else OUTPUT_DIR / 'dataset_view'
for split in SPLITS:
    image_dir = DATA_DIR / original_yaml[split]
    label_dir = DATA_DIR / original_yaml[split].replace('images', 'labels', 1)
    for kind, target in [('images', image_dir), ('labels', label_dir)]:
        link = VIEW_DIR / kind / split
        link.parent.mkdir(parents=True, exist_ok=True)
        if link.is_symlink() or link.exists():
            link.unlink()
        link.symlink_to(target.resolve(), target_is_directory=True)

manifest = pd.read_csv(DATA_DIR / 'frame_manifest.csv')
manifest['image'] = manifest['image_path'].map(lambda p: Path(p).stem)
manifest['image_file'] = [str(VIEW_DIR / 'images' / s / Path(p).name) for s, p in zip(manifest['split'], manifest['image_path'])]
manifest['label_file'] = [str(VIEW_DIR / 'labels' / s / f'{i}.txt') for s, i in zip(manifest['split'], manifest['image'])]
missing = [p for p in manifest['image_file'] if not Path(p).exists()]
assert not missing, f'{len(missing)} manifest images not found, e.g. {missing[:3]}'


test_df = manifest[manifest['split'] == 'test']
if SMOKE_TEST:
    n = min(SMOKE_TEST_IMAGES_PER_WEATHER, test_df['weather'].value_counts().min())
    test_df = test_df.groupby('weather').sample(n=n, random_state=SEED)
test_df = test_df.reset_index(drop=True)
WEATHERS = sorted(test_df['weather'].unique())

LIST_DIR = OUTPUT_DIR / 'image_lists'
LIST_DIR.mkdir(exist_ok=True)

def write_list(images, name):
    list_file = LIST_DIR / f'{name}.txt'
    list_file.write_text('\n'.join(images) + '\n')
    return str(list_file)

TRAIN_LIST = write_list(manifest.loc[manifest['split'] == 'train', 'image_file'], 'train')
VAL_LIST = write_list(manifest.loc[manifest['split'] == 'val', 'image_file'], 'val')

def make_yaml(test_images, name):
    yaml_file = OUTPUT_DIR / f'{name}.yaml'
    with open(yaml_file, 'w') as f:
        yaml.safe_dump({'path': str(VIEW_DIR), 'train': TRAIN_LIST, 'val': VAL_LIST,
                        'test': write_list(test_images, name), 'names': CLASS_NAMES}, f, sort_keys=False)
    return str(yaml_file)

RUNTIME_YAML = make_yaml(test_df['image_file'].tolist(), 'data_runtime')
WEATHER_YAMLS = {w: make_yaml(g['image_file'].tolist(), f'data_test_{w}') for w, g in test_df.groupby('weather')}

print('Classes:', CLASS_NAMES)
print('Runtime yaml:', RUNTIME_YAML)
display(pd.crosstab(manifest['weather'], manifest['split'], margins=True))
print(f'Evaluating on {len(test_df)} test images:', test_df['weather'].value_counts().to_dict())

Classes: ['car', 'bus', 'van', 'others']
Runtime yaml: /content/drive/MyDrive/DATA255/outputs/faster_rcnn/data_runtime.yaml


split,test,train,val,All
weather,,,,
cloudy,402,3468,649,4519
night,521,3407,657,4585
rainy,800,2583,637,4020
sunny,766,2342,595,3703
All,2489,11800,2538,16827


Evaluating on 2489 test images: {'rainy': 800, 'sunny': 766, 'night': 521, 'cloudy': 402}


Use the shared scoring helpers from the baseline.

In [ ]:
BOX_COLS = ['x1', 'y1', 'x2', 'y2']


def load_ground_truth(images_df):
    # Reads the YOLO label files and return one row per box in pixel coordinates.
    rows = []
    for r in images_df.itertuples():
        width, height = Image.open(r.image_file).size
        for line in Path(r.label_file).read_text().splitlines():
            if not line.strip():
                continue
            class_id, xc, yc, w, h = line.split()
            xc, w = float(xc) * width, float(w) * width
            yc, h = float(yc) * height, float(h) * height
            rows.append({'image': r.image, 'class_id': int(class_id),
                         'x1': xc - w / 2, 'y1': yc - h / 2, 'x2': xc + w / 2, 'y2': yc + h / 2})
    return pd.DataFrame(rows, columns=['image', 'class_id'] + BOX_COLS)


def predict_detections(model, images_df, conf, class_map=None):
    # Predicts boxes for the given images and returns one row per box in pixel coordinates.
    rows = []
    paths = images_df['image_file'].tolist()
    for start in range(0, len(paths), BATCH):
        results = model.predict(paths[start:start + BATCH], conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False)
        for result in results:
            image = Path(result.path).stem
            boxes = result.boxes
            for box, cls, score in zip(boxes.xyxy.tolist(), boxes.cls.int().tolist(), boxes.conf.tolist()):
                if class_map is not None:
                    if cls not in class_map:
                        continue
                    cls = class_map[cls]
                rows.append({'image': image, 'class_id': cls, 'confidence': score,
                             'x1': box[0], 'y1': box[1], 'x2': box[2], 'y2': box[3]})
        del results
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows, columns=['image', 'class_id', 'confidence'] + BOX_COLS)


def coco_ap(gt, det, image_names, agnostic=False):
    # AP with pycocotools. Returns {name: (AP50, AP50_95)} per class plus 'all' (mean over classes).
    # Classes with no ground truth in these images get NaN. agnostic=True -> one 'vehicle' class.
    image_ids = {name: i + 1 for i, name in enumerate(image_names)}
    gt = gt[gt['image'].isin(image_ids)]
    det = det[det['image'].isin(image_ids)]
    cat_names = ['vehicle'] if agnostic else CLASS_NAMES
    to_cat = (lambda c: 1) if agnostic else (lambda c: int(c) + 1)   # COCO category IDs start at 1

    coco_gt = COCO()
    coco_gt.dataset = {
        'images': [{'id': i} for i in image_ids.values()],
        'categories': [{'id': k + 1, 'name': n} for k, n in enumerate(cat_names)],
        'annotations': [{'id': k + 1, 'image_id': image_ids[r.image], 'category_id': to_cat(r.class_id),
                         'bbox': [r.x1, r.y1, r.x2 - r.x1, r.y2 - r.y1],
                         'area': (r.x2 - r.x1) * (r.y2 - r.y1), 'iscrowd': 0}
                        for k, r in enumerate(gt.itertuples())],
    }
    has_gt = {n: (gt['class_id'] == k).any() if not agnostic else len(gt) > 0 for k, n in enumerate(cat_names)}

    if len(det) == 0:   # no detections at all: AP is 0 wherever there is ground truth
        out = {n: (0.0, 0.0) if has_gt[n] else (np.nan, np.nan) for n in cat_names}
    else:
        with contextlib.redirect_stdout(io.StringIO()):   # hide pycocotools' printouts
            coco_gt.createIndex()
            coco_det = coco_gt.loadRes([{'image_id': image_ids[r.image], 'category_id': to_cat(r.class_id),
                                         'bbox': [r.x1, r.y1, r.x2 - r.x1, r.y2 - r.y1], 'score': r.confidence}
                                        for r in det.itertuples()])
            ev = COCOeval(coco_gt, coco_det, 'bbox')
            ev.params.maxDets = [1, 10, 300]   # allow up to 300 boxes per image (busy traffic scenes)
            ev.evaluate()
            ev.accumulate()
        # precision has shape [IoU thresholds (0.5..0.95), recall points, class, area range, maxDets]
        precision = ev.eval['precision'][:, :, :, 0, -1]
        out = {}
        for k, n in enumerate(cat_names):
            p = precision[:, :, k]
            out[n] = (p[0][p[0] > -1].mean(), p[p > -1].mean()) if has_gt[n] else (np.nan, np.nan)

    out['all'] = tuple(np.nanmean([out[n][j] for n in cat_names]) for j in (0, 1))
    return out


def box_iou(a, b):
    # IoU between every box in a (N, 4) and every box in b (M, 4), both in x1, y1, x2, y2 format.
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter)


def precision_recall(gt, det, agnostic=False):
    # Precision and recall at confidence >= COUNT_CONF and IoU >= IOU_MATCH.
    # Detections are matched greedily (highest confidence first) to unmatched GT boxes.
    # Returns {name: (precision, recall)}; precision is NaN for a class with no detections.
    det = det[det['confidence'] >= COUNT_CONF].sort_values('confidence', ascending=False).copy()
    det['tp'] = False
    key = ['image'] if agnostic else ['image', 'class_id']
    gt_by_key = dict(list(gt.groupby(key)))
    for k, d in det.groupby(key):
        g = gt_by_key.get(k)
        if g is None:
            continue
        ious = box_iou(d[BOX_COLS].to_numpy(), g[BOX_COLS].to_numpy())
        used = np.zeros(len(g), dtype=bool)
        for i, idx in enumerate(d.index):         # d keeps the confidence order
            candidates = np.where(~used & (ious[i] >= IOU_MATCH))[0]
            if len(candidates):
                used[candidates[np.argmax(ious[i, candidates])]] = True
                det.at[idx, 'tp'] = True

    def pr(d, n_gt):
        return (d['tp'].mean() if len(d) else np.nan, d['tp'].sum() / n_gt if n_gt else np.nan)

    if agnostic:
        return {'vehicle': pr(det, len(gt))}
    out = {n: pr(det[det['class_id'] == c], (gt['class_id'] == c).sum()) for c, n in enumerate(CLASS_NAMES)}
    out['all'] = pr(det, len(gt))
    return out


def evaluate(gt, det, images_df):
    # evaltuate all detection metrics for one model, as a long table: group, metric, value.
    image_names = images_df['image'].tolist()
    gt = gt[gt['image'].isin(image_names)]
    rows = []
    ap, pr = coco_ap(gt, det, image_names), precision_recall(gt, det)
    for group in ['all'] + CLASS_NAMES:
        rows += [(group, 'mAP50', ap[group][0]), (group, 'mAP50_95', ap[group][1]),
                 (group, 'precision', pr[group][0]), (group, 'recall', pr[group][1])]
    # mAP over only the classes that have a COCO equivalent (a fairer zero-shot comparison)
    mapped = [CLASS_NAMES[c] for c in sorted(set(COCO_TO_OURS.values()))]
    rows += [('coco_mapped_classes', m, np.nanmean([ap[n][j] for n in mapped])) for j, m in enumerate(['mAP50', 'mAP50_95'])]
    ap_v, pr_v = coco_ap(gt, det, image_names, agnostic=True)['vehicle'], precision_recall(gt, det, agnostic=True)['vehicle']
    rows += [('vehicle', 'mAP50', ap_v[0]), ('vehicle', 'mAP50_95', ap_v[1]),
             ('vehicle', 'precision', pr_v[0]), ('vehicle', 'recall', pr_v[1])]
    for weather, g in images_df.groupby('weather'):
        ap_w = coco_ap(gt, det, g['image'].tolist())['all']
        rows += [(f'weather:{weather}', 'mAP50', ap_w[0]), (f'weather:{weather}', 'mAP50_95', ap_w[1])]
    return pd.DataFrame(rows, columns=['group', 'metric', 'value'])


def count_table(gt, det, images_df):
    # Count the number of boxes per image, per class, and overall, for both ground truth and predictions.
    det = det[det['confidence'] >= COUNT_CONF]
    counts = images_df[['image', 'sequence', 'frame', 'weather']].copy()
    for prefix, boxes in [('gt', gt), ('pred', det)]:
        per_class = (boxes.groupby(['image', 'class_id']).size().unstack(fill_value=0)
                     .reindex(index=counts['image'], columns=range(NUM_CLASSES), fill_value=0))
        for c, n in enumerate(CLASS_NAMES):
            counts[f'{prefix}_{n}'] = per_class[c].to_numpy()
        counts[f'{prefix}_total'] = per_class.sum(axis=1).to_numpy()
    return counts


def count_mae(counts):
    # Mean absolute counting error per image: overall, per class and per weather.
    err = lambda d, n: (d[f'pred_{n}'] - d[f'gt_{n}']).abs().mean()
    rows = [('all', 'count_MAE', err(counts, 'total'))]
    rows += [(n, 'count_MAE', err(counts, n)) for n in CLASS_NAMES]
    rows += [(f'weather:{w}', 'count_MAE', err(g, 'total')) for w, g in counts.groupby('weather')]
    return pd.DataFrame(rows, columns=['group', 'metric', 'value'])


test_gt = load_ground_truth(test_df)
print(f'{len(test_gt)} ground-truth boxes in {len(test_df)} test images')

13221 ground-truth boxes in 2489 test images


## 3. DATA AND MODEL

Prepare Faster R-CNN inputs and use the COCO pretrained ResNet-50 model.

In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.v2.functional as TF
from torchvision.transforms import ColorJitter

torch.manual_seed(SEED)
FR_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')


def read_labels(label_file, width, height):
    # YOLO label file -> (N, 4) pixel boxes and (N,) class ids, exactly like load_ground_truth
    boxes, labels = [], []
    for line in Path(label_file).read_text().splitlines():
        if not line.strip():
            continue
        c, xc, yc, w, h = line.split()
        xc, w, yc, h = float(xc) * width, float(w) * width, float(yc) * height, float(h) * height
        boxes.append([xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2])
        labels.append(int(c))
    return (torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4), torch.tensor(labels, dtype=torch.int64))


class DetracDataset(torch.utils.data.Dataset):
    def __init__(self, images_df, train):
        self.df, self.train = images_df.reset_index(drop=True), train
        self.jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['image_file']).convert('RGB')
        boxes, labels = read_labels(r['label_file'], *img.size)
        if self.train:
            img = self.jitter(img)
        img = TF.to_dtype(TF.to_image(img), torch.float32, scale=True)
        if self.train and random.random() < 0.5:
            img = img.flip(-1)
            w = img.shape[-1]
            boxes = torch.stack([w - boxes[:, 2], boxes[:, 1], w - boxes[:, 0], boxes[:, 3]], 1) if len(boxes) else boxes
        keep = (boxes[:, 2] > boxes[:, 0] + 1) & (boxes[:, 3] > boxes[:, 1] + 1)
        # +1: torchvision reserves label 0 for background
        return img, {'boxes': boxes[keep], 'labels': labels[keep] + 1}, r['image']


def collate(batch):
    return tuple(zip(*batch))


def build_model():
    model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1,
                                       min_size=MIN_SIZE, max_size=MAX_SIZE,
                                       box_score_thresh=EVAL_CONF, box_detections_per_img=300)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES + 1)   # 4 classes + background
    return model.to(FR_DEVICE)


@torch.no_grad()
def predict_frcnn(model, images_df, conf=EVAL_CONF, batch=BATCH):
    # Same output as predict_detections: one row per box, pixel coordinates, class ids 0..3
    model.eval()
    loader = torch.utils.data.DataLoader(DetracDataset(images_df, train=False), batch_size=batch,
                                         num_workers=WORKERS, collate_fn=collate)
    rows = []
    for imgs, _, names in loader:
        outs = model([im.to(FR_DEVICE) for im in imgs])
        for name, o in zip(names, outs):
            keep = o['scores'] >= conf
            for b, l, s in zip(o['boxes'][keep].tolist(), o['labels'][keep].tolist(), o['scores'][keep].tolist()):
                rows.append({'image': name, 'class_id': l - 1, 'confidence': s, 'x1': b[0], 'y1': b[1], 'x2': b[2], 'y2': b[3]})
    return pd.DataFrame(rows, columns=['image', 'class_id', 'confidence'] + BOX_COLS)


train_df = manifest[manifest['split'] == 'train']
if FRACTION < 1:
    train_df = train_df.sample(frac=FRACTION, random_state=SEED)
if MAX_TRAIN_IMAGES:
    train_df = train_df.sample(n=min(MAX_TRAIN_IMAGES, len(train_df)), random_state=SEED)
val_sub = manifest[manifest['split'] == 'val'].sample(n=VAL_IMAGES, random_state=SEED).reset_index(drop=True)
val_sub_gt = load_ground_truth(val_sub)
print(f'device {FR_DEVICE} | train images {len(train_df)} | val subset {len(val_sub)} | epochs {EPOCHS}')

device cuda | train images 11800 | val subset 600 | epochs 12


## 4. FINE TUNE

Train each epoch and keep the checkpoint with the best validation score.

In [ ]:
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LAST_PT, BEST_PT, DONE_FILE = CKPT_DIR / 'last.pt', CKPT_DIR / 'best.pt', OUTPUT_DIR / 'TRAINING_DONE'

model = build_model()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
loader = torch.utils.data.DataLoader(DetracDataset(train_df, train=True), batch_size=BATCH, shuffle=True,
                                     num_workers=WORKERS, collate_fn=collate, drop_last=True,
                                     generator=torch.Generator().manual_seed(SEED))
WARMUP = min(500, len(loader))
lr_at = lambda step: LR * (min(1.0, (step + 1) / WARMUP)) * (0.1 ** sum(step >= m * len(loader) for m in (8, 11)))
scaler = torch.amp.GradScaler('cuda', enabled=FR_DEVICE.type == 'cuda')

state = {'epoch': 0, 'step': 0, 'best': -1.0, 'best_epoch': None, 'history': []}
if LAST_PT.exists():
    ck = torch.load(LAST_PT, map_location=FR_DEVICE, weights_only=False)
    model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); state = ck['state']
    print(f"Resuming after epoch {state['epoch']} (best val mAP50-95 {state['best']:.3f} at epoch {state['best_epoch']})")

start = time.time()
while state['epoch'] < EPOCHS and not DONE_FILE.exists():
    model.train()
    t0, running = time.time(), 0.0
    for imgs, targets, _ in loader:
        for g in optimizer.param_groups:
            g['lr'] = lr_at(state['step'])
        imgs = [im.to(FR_DEVICE) for im in imgs]
        targets = [{k: v.to(FR_DEVICE) for k, v in t.items()} for t in targets]
        with torch.autocast('cuda', enabled=FR_DEVICE.type == 'cuda'):
            loss = sum(model(imgs, targets).values())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        state['step'] += 1
    train_sec = time.time() - t0
    det = predict_frcnn(model, val_sub)
    val_map50, val_map = coco_ap(val_sub_gt, det, val_sub['image'].tolist())['all']
    state['epoch'] += 1
    state['history'].append({'epoch': state['epoch'], 'train_loss': running / len(loader), 'val_mAP50': val_map50,
                             'val_mAP50_95': val_map, 'train_seconds': train_sec, 'lr': optimizer.param_groups[0]['lr']})
    if val_map > state['best']:
        state['best'], state['best_epoch'] = val_map, state['epoch']
        torch.save(model.state_dict(), BEST_PT)
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'state': state}, LAST_PT)
    pd.DataFrame(state['history']).to_csv(OUTPUT_DIR / 'training_log.csv', index=False)
    print(f"epoch {state['epoch']}/{EPOCHS}  loss {running / len(loader):.3f}  val mAP50 {val_map50:.3f}  "
          f"mAP50-95 {val_map:.3f}  ({train_sec / 60:.1f} min train)")
    if SMOKE_TEST:
        full = (manifest['split'] == 'train').sum()
        print(f'SMOKE TEST: {train_sec / len(train_df):.3f} s per training image -> one full epoch '
              f'({full} images) takes about {train_sec / len(train_df) * full / 60:.0f} min; the real run is 12 epochs.')

if not DONE_FILE.exists() and state['epoch'] >= EPOCHS:
    DONE_FILE.write_text(json.dumps({'epochs_completed': state['epoch'], 'best_epoch': state['best_epoch'],
                                     'best_val_mAP50_95_subset': state['best']}))
print('Training status:', DONE_FILE.read_text() if DONE_FILE.exists() else 'not finished')
display(pd.DataFrame(state['history']).round(3))

Resuming after epoch 8 (best val mAP50-95 0.645 at epoch 1)
epoch 9/12  loss 0.133  val mAP50 0.865  mAP50-95 0.630  (23.6 min train)
epoch 10/12  loss 0.123  val mAP50 0.864  mAP50-95 0.631  (24.0 min train)
epoch 11/12  loss 0.117  val mAP50 0.860  mAP50-95 0.633  (24.0 min train)
epoch 12/12  loss 0.112  val mAP50 0.861  mAP50-95 0.633  (24.0 min train)
Training status: {"epochs_completed": 12, "best_epoch": 1, "best_val_mAP50_95_subset": 0.6449282776529992}


,epoch,train_loss,val_mAP50,val_mAP50_95,train_seconds,lr
0,1,0.326,0.883,0.645,1421.696,0.010
1,2,0.221,0.879,0.644,1425.983,0.010
2,3,0.211,0.867,0.638,1433.324,0.010
3,4,0.196,0.869,0.615,1434.413,0.010
4,5,0.183,0.826,0.566,1432.833,0.010
5,6,0.175,0.872,0.612,1434.640,0.010
6,7,0.169,0.802,0.585,1435.108,0.010
7,8,0.163,0.855,0.620,1432.607,0.010
8,9,0.133,0.865,0.630,1416.711,0.001
9,10,0.123,0.864,0.631,1437.705,0.001


## 5. EVALUATION AND DETECTIONS EXPORT

Evaluate the best checkpoint and save val/test detections for the comparison.

In [ ]:
model = build_model()
model.load_state_dict(torch.load(BEST_PT, map_location=FR_DEVICE))
model.eval()
test_gt = load_ground_truth(test_df)
val_df = manifest[manifest['split'] == 'val']
if SMOKE_TEST:
    n = min(SMOKE_TEST_IMAGES_PER_WEATHER, val_df['weather'].value_counts().min())
    val_df = val_df.groupby('weather').sample(n=n, random_state=SEED)
val_df = val_df.reset_index(drop=True)

start = time.time()
frcnn_det = predict_frcnn(model, test_df)
frcnn_val_det = predict_frcnn(model, val_df)
print(f'Prediction (val + test): {time.time() - start:.0f}s')


def save_boxes(boxes, images_df, name):
    # Same as the baseline notebook: adds image info and writes one row per box.
    info = images_df[['image', 'sequence', 'frame', 'weather']].copy()
    sizes = [Image.open(f).size for f in images_df['image_file']]
    info['image_width'], info['image_height'] = [s[0] for s in sizes], [s[1] for s in sizes]
    out = boxes.merge(info, on='image', how='left')
    out['class_name'] = out['class_id'].map(dict(enumerate(CLASS_NAMES)))
    cols = ['image', 'sequence', 'frame', 'weather', 'image_width', 'image_height', 'class_id', 'class_name']
    out = out[cols + [c for c in ['confidence'] if c in out] + BOX_COLS]
    out.to_csv(OUTPUT_DIR / name, index=False)
    print(f'{len(out):>7} boxes in {len(images_df)} images ->', OUTPUT_DIR / name)


save_boxes(frcnn_val_det, val_df, 'frcnn_detections_val.csv')
save_boxes(frcnn_det, test_df, 'frcnn_detections_test.csv')

frcnn_metrics = evaluate(test_gt, frcnn_det, test_df)
frcnn_counts = count_table(test_gt, frcnn_det, test_df)
frcnn_count_mae = count_mae(frcnn_counts)
frcnn_all = pd.concat([frcnn_metrics, frcnn_count_mae])
frcnn_all.to_csv(OUTPUT_DIR / 'frcnn_metrics.csv', index=False)
with pd.option_context('display.max_rows', 200):
    display(frcnn_all.pivot(index='group', columns='metric', values='value').round(3))

Prediction (val + test): 655s
 240606 boxes in 2538 images -> /content/drive/MyDrive/DATA255/outputs/faster_rcnn/frcnn_detections_val.csv
 140777 boxes in 2489 images -> /content/drive/MyDrive/DATA255/outputs/faster_rcnn/frcnn_detections_test.csv


metric,count_MAE,mAP50,mAP50_95,precision,recall
group,,,,,
all,1.616,0.737,0.567,0.706,0.894
bus,0.308,0.909,0.716,0.736,0.898
car,1.063,0.905,0.665,0.756,0.903
coco_mapped_classes,NaN,0.756,0.584,NaN,NaN
others,0.045,0.452,0.372,0.198,0.489
van,0.464,0.682,0.514,0.315,0.773
vehicle,NaN,0.914,0.680,0.722,0.915
weather:cloudy,1.918,0.783,0.502,NaN,NaN
weather:night,1.440,0.636,0.466,NaN,NaN


## 6. SPEED

Measure FPS at batch 1 and batch throughput on the test set.

In [ ]:
def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elif torch.backends.mps.is_available():
        torch.mps.synchronize()


@torch.no_grad()
def measure_fps_frcnn(model, paths, batch):
    model.eval()
    load = lambda p: TF.to_dtype(TF.to_image(Image.open(p).convert('RGB')), torch.float32, scale=True).to(FR_DEVICE)
    model([load(p) for p in paths[:batch]])   # warm-up
    sync()
    start = time.perf_counter()
    for i in range(0, len(paths), batch):
        outs = model([load(p) for p in paths[i:i + batch]])
        _ = [o['boxes'][o['scores'] >= COUNT_CONF].cpu() for o in outs]
    sync()
    return len(paths) / (time.perf_counter() - start)


fps_paths = test_df.sample(n=min(300, len(test_df)), random_state=SEED)['image_file'].tolist()
for p in fps_paths:   # read every image once, as in the P2 notebook, so timing excludes cold disk reads
    Image.open(p).load()
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(FR_DEVICE)
speed = pd.DataFrame([{'model': 'faster_rcnn_r50_fpn_v2', 'device': device_name, 'batch': b, 'images': len(fps_paths),
                       'fps': measure_fps_frcnn(model, fps_paths, b)} for b in [1, BATCH]])
speed['ms_per_image'] = 1000 / speed['fps']
speed['params_M'] = sum(p.numel() for p in model.parameters()) / 1e6
speed.to_csv(OUTPUT_DIR / 'frcnn_speed.csv', index=False)
print('Saved', OUTPUT_DIR / 'frcnn_speed.csv')
display(speed.round(1))

Saved /content/drive/MyDrive/DATA255/outputs/faster_rcnn/frcnn_speed.csv


,model,device,batch,images,fps,ms_per_image,params_M
0,faster_rcnn_r50_fpn_v2,Tesla T4,1,300,8.8,113.5,43.3
1,faster_rcnn_r50_fpn_v2,Tesla T4,4,300,8.4,118.9,43.3


Save the final settings and main results.

In [ ]:
r = frcnn_all.set_index(['group', 'metric'])['value']
summary = {
    'model': MODEL, 'weights': str(BEST_PT.relative_to(OUTPUT_DIR)), 'num_outputs': NUM_CLASSES + 1,
    'best_epoch': state['best_epoch'], 'epochs_trained': state['epoch'],
    'best_val_subset_mAP50_95': state['best'],
    'settings': {'epochs': EPOCHS, 'batch': BATCH, 'lr': LR, 'min_size': MIN_SIZE, 'max_size': MAX_SIZE,
                 'train_images': len(train_df), 'val_subset_images': len(val_sub), 'seed': SEED,
                 'eval_conf': EVAL_CONF, 'count_conf': COUNT_CONF, 'iou_match': IOU_MATCH,
                 'augmentation': 'horizontal flip p=0.5, colour jitter 0.2', 'torchvision': torchvision.__version__},
    'test': {'mAP50_4_classes': float(r[('all', 'mAP50')]), 'mAP50_95_4_classes': float(r[('all', 'mAP50_95')]),
             'vehicle_mAP50': float(r[('vehicle', 'mAP50')]), 'vehicle_recall': float(r[('vehicle', 'recall')]),
             'count_MAE': float(r[('all', 'count_MAE')])},
    'speed': {'device': device_name, **{f'fps_batch_{b}': float(f) for b, f in zip(speed['batch'], speed['fps'])}},
}
(OUTPUT_DIR / 'final_model_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('Detailed day/night, size and occlusion tables: run compare_detectors.ipynb')

{
  "model": "fasterrcnn_resnet50_fpn_v2",
  "weights": "checkpoints/best.pt",
  "num_outputs": 5,
  "best_epoch": 1,
  "epochs_trained": 12,
  "best_val_subset_mAP50_95": 0.6449282776529992,
  "settings": {
    "epochs": 12,
    "batch": 4,
    "lr": 0.01,
    "min_size": 540,
    "max_size": 960,
    "train_images": 11800,
    "val_subset_images": 600,
    "seed": 0,
    "eval_conf": 0.001,
    "count_conf": 0.25,
    "iou_match": 0.5,
    "augmentation": "horizontal flip p=0.5, colour jitter 0.2",
    "torchvision": "0.29.1+cu130"
  },
  "test": {
    "mAP50_4_classes": 0.7372685302079096,
    "mAP50_95_4_classes": 0.5666836596285735,
    "vehicle_mAP50": 0.9137477019722344,
    "vehicle_recall": 0.9149081007488087,
    "count_MAE": 1.6159100040176777
  },
  "speed": {
    "device": "Tesla T4",
    "fps_batch_1": 8.813390409413568,
    "fps_batch_4": 8.409457269185882
  }
}
Detailed day/night, size and occlusion tables: run compare_detectors.ipynb
